In [ ]:
# CELL 1 — MOUNT / INSTALL / MANIFEST-BOUND RESUME
from google.colab import drive
drive.mount('/content/drive')

from datetime import datetime, timezone
import hashlib, importlib.util, json, os, shutil, subprocess, sys, time, uuid
from pathlib import Path

DRIVE_LANE = Path('/content/drive/MyDrive/private-cell-tracking/division_temporal_colab_v1')
SOURCE_SNAPSHOT = DRIVE_LANE / 'source_snapshot'
SOURCE_MANIFEST = SOURCE_SNAPSHOT / 'SOURCE_MANIFEST.json'
CARRIER = DRIVE_LANE / 'carrier'
PERSISTENT_STATE = DRIVE_LANE / 'checkpoint_state'
COLAB_LEDGER = DRIVE_LANE / 'COLAB_HOURS_LEDGER.jsonl'

def canonical_bytes(value):
    return (json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=True) + '\n').encode('ascii')

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def verify_source_snapshot(root, manifest_path):
    manifest = json.loads(manifest_path.read_text(encoding='ascii'))
    rows = manifest.get('files')
    if manifest.get('schema') != 'celltrack-alpha.colab-source-snapshot.v1' or not isinstance(rows, list):
        raise RuntimeError('invalid source snapshot manifest')
    observed = []
    for row in rows:
        relative = Path(row['path'])
        if relative.is_absolute() or '..' in relative.parts or relative.as_posix() == 'SOURCE_MANIFEST.json':
            raise RuntimeError('unsafe source snapshot path')
        path = root / relative
        current = {'path': relative.as_posix(), 'bytes': path.stat().st_size, 'sha256': file_sha256(path)}
        if current != row:
            raise RuntimeError(f'source snapshot mismatch: {relative}')
        observed.append(current)
    declared = {row['path'] for row in rows} | {'SOURCE_MANIFEST.json'}
    actual = {path.relative_to(root).as_posix() for path in root.rglob('*') if path.is_file()}
    if actual != declared:
        raise RuntimeError('source snapshot has undeclared or missing files')
    tree_sha = hashlib.sha256(canonical_bytes(observed)).hexdigest()
    if tree_sha != manifest.get('tree_sha256'):
        raise RuntimeError('source snapshot tree digest mismatch')
    return manifest

SOURCE_RECEIPT = verify_source_snapshot(SOURCE_SNAPSHOT, SOURCE_MANIFEST)
LOCAL_REPO = Path('/content') / f"cell_tracking_{SOURCE_RECEIPT['tree_sha256'][:12]}"
if not LOCAL_REPO.exists():
    for row in SOURCE_RECEIPT['files']:
        destination = LOCAL_REPO / row['path']
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(SOURCE_SNAPSHOT / row['path'], destination)
    shutil.copy2(SOURCE_MANIFEST, LOCAL_REPO / 'SOURCE_MANIFEST.json')
verify_source_snapshot(LOCAL_REPO, SOURCE_MANIFEST)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(LOCAL_REPO / 'reports/g02/screening/tiered_surface/branches/division_temporal_colab_v1/COLAB_REQUIREMENTS.txt')], check=True)
sys.path.insert(0, str(LOCAL_REPO))
os.chdir(LOCAL_REPO)

from scripts.colab_division_harness import (
    AtomicCheckpointStore, ColabSessionGuard, FullStageLogger, SwapGuard,
    adjudicate_resume_runtime, append_colab_session_start, assigned_gpu_model,
    canonical_json_bytes, publish_training_checkpoint, reconcile_interrupted_colab_sessions,
    restore_training_state,
    runtime_fingerprint, sha256_bytes, sha256_file, validate_objective_contract,
)
from scripts.preprocess_division_colab_data import verify_output

SESSION_ID = f"colab-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex[:8]}"
SESSION_STARTED = datetime.now(timezone.utc).isoformat()
CURRENT_RUNTIME = runtime_fingerprint()
ASSIGNED_GPU = assigned_gpu_model()
HARDWARE_SHA = sha256_bytes(canonical_json_bytes(CURRENT_RUNTIME))
START_RECEIPT_SHA = sha256_bytes(canonical_json_bytes({'session_id': SESSION_ID, 'started_utc': SESSION_STARTED, 'assigned_gpu_model': ASSIGNED_GPU, 'hardware_fingerprint_sha256': HARDWARE_SHA}))
append_colab_session_start(COLAB_LEDGER, session_id=SESSION_ID, started_utc=SESSION_STARTED, assigned_gpu_model=ASSIGNED_GPU, hardware_fingerprint_sha256=HARDWARE_SHA, receipt_sha256=START_RECEIPT_SHA)

BRANCH = LOCAL_REPO / 'reports/g02/screening/tiered_surface/branches/division_temporal_colab_v1'
OBJECTIVE_PATH = BRANCH / 'TRAINING_OBJECTIVE_TEMPLATE.json'
OBJECTIVE = validate_objective_contract(json.loads(OBJECTIVE_PATH.read_text(encoding='ascii')))
CARRIER_RECEIPT = verify_output(CARRIER)
if (CARRIER_RECEIPT['video_count'], CARRIER_RECEIPT['counts']['frames'], CARRIER_RECEIPT['counts']['parent_events'], CARRIER_RECEIPT['counts']['division_edges']) != (199, 19900, 151, 302):
    raise RuntimeError('competition carrier counts drifted')
ENTRYPOINT = LOCAL_REPO / OBJECTIVE['training']['entrypoint']
if sha256_file(ENTRYPOINT) != OBJECTIVE['training']['entrypoint_sha256']:
    raise RuntimeError('training entrypoint hash mismatch')

STORE = AtomicCheckpointStore(PERSISTENT_STATE)
REMOTE_LATEST = STORE.verify_latest()
EXPECTED_MANIFEST_SHA = REMOTE_LATEST['pointer']['manifest_sha256']
LOCAL_STATE = Path('/content') / f"division_parent_state_{EXPECTED_MANIFEST_SHA[:12]}"
PULL = STORE.pull_latest_verified(LOCAL_STATE, expected_manifest_sha256=EXPECTED_MANIFEST_SHA)
RUNTIME_CONTINUITY = adjudicate_resume_runtime(PULL['semantic']['producer_runtime'], CURRENT_RUNTIME, allow_gpu_transition=True)
RECONCILED_SESSIONS = reconcile_interrupted_colab_sessions(COLAB_LEDGER, before_utc=SESSION_STARTED, exclude_session_id=SESSION_ID, latest_producer_session=PULL['producer_session'], checkpoint_cadence_seconds=600.0, session_limit_seconds=6.0 * 3600.0)

SESSION_LOG = Path('/content') / f'{SESSION_ID}.full.jsonl'
shutil.copy2(PULL['log_path'], SESSION_LOG)
LOGGER = FullStageLogger(SESSION_LOG, run_id=SESSION_ID, config=OBJECTIVE)
LOGGER.emit('session_start_verified', assigned_gpu_model=ASSIGNED_GPU, hardware_fingerprint_sha256=HARDWARE_SHA, runtime_continuity=RUNTIME_CONTINUITY, reconciled_interrupted_sessions=len(RECONCILED_SESSIONS), source_tree_sha256=SOURCE_RECEIPT['tree_sha256'], input_manifest_sha256=CARRIER_RECEIPT['manifest_sha256'], checkpoint_manifest_sha256=EXPECTED_MANIFEST_SHA, checkpoint_state_digest=PULL['semantic']['state_digest'], examined_count=19900, modified_count=0)
SESSION_TARGET_HOURS = float(OBJECTIVE['training']['session_target_hours'])
GUARD = ColabSessionGuard(checkpoint_interval_seconds=600.0, session_limit_seconds=SESSION_TARGET_HOURS * 3600.0, safety_margin_seconds=900.0)
SWAP_GUARD = SwapGuard()
SESSION_META = {'session_id': SESSION_ID, 'started_utc': SESSION_STARTED, 'assigned_gpu_model': ASSIGNED_GPU, 'hardware_fingerprint_sha256': HARDWARE_SHA}
NEXT_PUBLICATION = int(PULL['pointer']['publication_index']) + 1
PUBLICATIONS = []

def push_checkpoint(**state):
    global NEXT_PUBLICATION
    SWAP_GUARD.sample()
    checkpoint_started_utc = datetime.now(timezone.utc).isoformat()
    LOGGER.emit('checkpoint_publish_start', publication_index=NEXT_PUBLICATION, global_step=int(state['global_step']), examined_count=0, modified_count=0)
    checkpoint_session = {**SESSION_META, 'checkpoint_started_utc': checkpoint_started_utc}
    receipt = publish_training_checkpoint(store=STORE, publication_index=NEXT_PUBLICATION, log_path=SESSION_LOG, session=checkpoint_session, **state)
    NEXT_PUBLICATION += 1
    GUARD.mark_checkpoint()
    PUBLICATIONS.append({'at_monotonic': time.monotonic(), 'receipt': receipt})
    return receipt

def restore_checkpoint(*, model, optimizer, scheduler, scaler):
    return restore_training_state(PULL['checkpoint_path'].read_bytes(), model=model, optimizer=optimizer, scheduler=scheduler, scaler=scaler, expected_provenance=PULL['semantic']['provenance'], runtime_policy=OBJECTIVE['training']['resume_hardware_policy'])

COLAB_CONTEXT = {'objective': OBJECTIVE, 'carrier_root': CARRIER, 'carrier_receipt': CARRIER_RECEIPT, 'checkpoint_path': PULL['checkpoint_path'], 'checkpoint_semantic_receipt': PULL['semantic'], 'runtime_continuity': RUNTIME_CONTINUITY, 'restore_checkpoint': restore_checkpoint, 'logger': LOGGER, 'guard': GUARD, 'swap_guard': SWAP_GUARD, 'checkpoint_due': GUARD.checkpoint_due, 'graceful_yield_due': GUARD.graceful_yield_due, 'push_checkpoint': push_checkpoint, 'session': SESSION_META}
print(json.dumps({'status': 'READY_TO_RESUME', 'session_id': SESSION_ID, 'assigned_gpu_model': ASSIGNED_GPU, 'runtime_continuity': RUNTIME_CONTINUITY, 'checkpoint_manifest_sha256': EXPECTED_MANIFEST_SHA, 'semantic_state_digest': PULL['semantic']['state_digest']}, indent=2, sort_keys=True))


In [ ]:
# CELL 2 — TRAIN / TEN-MINUTE DURABILITY CADENCE
spec = importlib.util.spec_from_file_location('division_parent_training_entrypoint', ENTRYPOINT)
if spec is None or spec.loader is None:
    raise RuntimeError('cannot load the frozen training entrypoint')
training_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(training_module)
if not callable(getattr(training_module, 'run_colab_session', None)):
    raise RuntimeError('entrypoint must export run_colab_session(context)')
with LOGGER.stage('train_session', counts={'examined': 19900, 'modified': 0}):
    TRAIN_RESULT = training_module.run_colab_session(COLAB_CONTEXT)
required = {'model', 'optimizer', 'scheduler', 'scaler', 'sampler_state', 'global_step', 'epoch', 'provenance', 'maximum_uncheckpointed_seconds'}
if not isinstance(TRAIN_RESULT, dict) or not required.issubset(TRAIN_RESULT):
    raise RuntimeError(f'training result lacks durability fields: {sorted(required - set(TRAIN_RESULT or {}))}')
if float(TRAIN_RESULT['maximum_uncheckpointed_seconds']) > 600.0:
    raise RuntimeError('training exceeded the ten-minute durability contract')
if GUARD.graceful_yield_due() and not PUBLICATIONS:
    raise RuntimeError('session reached its yield boundary without a remote checkpoint')
print(json.dumps({'status': 'TRAIN_CELL_COMPLETE', 'global_step': int(TRAIN_RESULT['global_step']), 'epoch': int(TRAIN_RESULT['epoch']), 'periodic_publications': len(PUBLICATIONS), 'maximum_uncheckpointed_seconds': float(TRAIN_RESULT['maximum_uncheckpointed_seconds'])}, indent=2, sort_keys=True))


In [ ]:
# CELL 3 — FINAL CHECKPOINT / FULL JSONL / DEVICE-SPECIFIC LEDGER
from scripts.colab_division_harness import append_colab_session, colab_hours_by_gpu_model, write_idempotent_json
FINAL_RECEIPT_PATH = DRIVE_LANE / 'session_receipts' / f'{SESSION_ID}.json'
if FINAL_RECEIPT_PATH.exists():
    FINAL_SESSION_RECEIPT = json.loads(FINAL_RECEIPT_PATH.read_text(encoding='ascii'))
else:
    LOGGER.emit('session_terminal', status='COMPLETE_OR_GRACEFUL_YIELD', swap_guard=SWAP_GUARD.receipt(), periodic_checkpoint_count=len(PUBLICATIONS), examined_count=19900, modified_count=int(TRAIN_RESULT.get('modified_count', 0)))
    checkpoint_fields = {key: TRAIN_RESULT[key] for key in ('model', 'optimizer', 'scheduler', 'scaler', 'sampler_state', 'global_step', 'epoch', 'provenance')}
    FINAL_CHECKPOINT = push_checkpoint(**checkpoint_fields)
    SESSION_ENDED = datetime.now(timezone.utc).isoformat()
    receipt_body = {'schema': 'celltrack-alpha.colab-session-receipt.v1', 'session_id': SESSION_ID, 'started_utc': SESSION_STARTED, 'ended_utc': SESSION_ENDED, 'assigned_gpu_model': ASSIGNED_GPU, 'hardware_fingerprint_sha256': HARDWARE_SHA, 'final_checkpoint_manifest_sha256': FINAL_CHECKPOINT['manifest_sha256'], 'final_checkpoint_state_digest': FINAL_CHECKPOINT['state_digest'], 'full_jsonl_sha256': sha256_file(SESSION_LOG), 'source_tree_sha256': SOURCE_RECEIPT['tree_sha256'], 'input_manifest_sha256': CARRIER_RECEIPT['manifest_sha256'], 'status': 'COMPLETE_OR_GRACEFUL_YIELD'}
    FINAL_SESSION_RECEIPT = {**receipt_body, 'ledger_receipt_sha256': sha256_bytes(canonical_json_bytes(receipt_body))}
    write_idempotent_json(FINAL_RECEIPT_PATH, FINAL_SESSION_RECEIPT)
append_colab_session(COLAB_LEDGER, session_id=SESSION_ID, started_utc=FINAL_SESSION_RECEIPT['started_utc'], ended_utc=FINAL_SESSION_RECEIPT['ended_utc'], assigned_gpu_model=FINAL_SESSION_RECEIPT['assigned_gpu_model'], hardware_fingerprint_sha256=FINAL_SESSION_RECEIPT['hardware_fingerprint_sha256'], status=FINAL_SESSION_RECEIPT['status'], receipt_sha256=FINAL_SESSION_RECEIPT['ledger_receipt_sha256'], notes='Hours remain bucketed by exact assigned GPU model; Kaggle hours excluded.')
print(json.dumps({'status': 'SESSION_PUBLISHED', 'receipt': FINAL_SESSION_RECEIPT, 'colab_hours_by_assigned_gpu_model': colab_hours_by_gpu_model(COLAB_LEDGER)}, indent=2, sort_keys=True))
